# 모델 동작 확인 (CPU)

BiEncoder + Loss가 올바르게 동작하는지 확인한다.
- `q_emb`, `p_emb` shape이 (B, 768)인지
- loss 초기값이 log(B) 근처인지
- 몇 step 후 loss가 감소하는지

In [ ]:
import sys
sys.path.append('..')

import math
import torch
from transformers import BertTokenizerFast
from src.models.biencoder import BiEncoder
from src.models.loss import in_batch_negative_loss

## 1. 더미 데이터 준비

In [ ]:
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

questions = [
    "Who built the Eiffel Tower?",
    "What is the capital of France?",
    "When was the French Revolution?",
    "Who wrote Les Misérables?",
]

positives = [
    "Eiffel Tower [SEP] The Eiffel Tower was designed and built by Gustave Eiffel.",
    "Paris [SEP] Paris is the capital and most populous city of France.",
    "French Revolution [SEP] The French Revolution began in 1789 and ended in 1799.",
    "Victor Hugo [SEP] Les Misérables is a novel by French author Victor Hugo.",
]

hard_negs = [
    "Eiffel Tower [SEP] The Eiffel Tower is located on the Champ de Mars in Paris.",
    "France [SEP] France is a country in Western Europe with a rich cultural history.",
    "French Revolution [SEP] The French Revolution led to the rise of Napoleon Bonaparte.",
    "Victor Hugo [SEP] Victor Hugo was born in Besançon, France, in 1802.",
]

def tokenize(texts, max_length):
    return tokenizer(texts, max_length=max_length, padding='max_length',
                     truncation=True, return_tensors='pt')

q_enc = tokenize(questions, max_length=64)
p_enc = tokenize(positives, max_length=256)
h_enc = tokenize(hard_negs, max_length=256)

print('q_input_ids shape:', q_enc['input_ids'].shape)  # (4, 64)
print('p_input_ids shape:', p_enc['input_ids'].shape)  # (4, 256)

## 2. BiEncoder forward pass — shape 확인

In [ ]:
model = BiEncoder()

q_emb, p_emb = model(
    q_enc['input_ids'], q_enc['attention_mask'], q_enc['token_type_ids'],
    p_enc['input_ids'], p_enc['attention_mask'], p_enc['token_type_ids'],
)
h_emb, _ = model(
    q_enc['input_ids'], q_enc['attention_mask'], q_enc['token_type_ids'],
    h_enc['input_ids'], h_enc['attention_mask'], h_enc['token_type_ids'],
)

print('q_emb shape:', q_emb.shape)  # (4, 768)
print('p_emb shape:', p_emb.shape)  # (4, 768)
print('h_emb shape:', h_emb.shape)  # (4, 768)

## 3. Loss 초기값 확인

random 초기화 상태에서 loss는 `log(B)` 근처여야 한다.  
B=4이면 log(4) ≈ 1.386

In [ ]:
loss = in_batch_negative_loss(q_emb, p_emb, h_emb)

B = len(questions)
print(f'loss:        {loss.item():.4f}')
print(f'log(B={B}): {math.log(B):.4f}')

## 4. Loss 감소 확인 — 몇 step 학습

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

for step in range(5):
    optimizer.zero_grad()

    q_emb, p_emb = model(
        q_enc['input_ids'], q_enc['attention_mask'], q_enc['token_type_ids'],
        p_enc['input_ids'], p_enc['attention_mask'], p_enc['token_type_ids'],
    )
    h_emb, _ = model(
        q_enc['input_ids'], q_enc['attention_mask'], q_enc['token_type_ids'],
        h_enc['input_ids'], h_enc['attention_mask'], h_enc['token_type_ids'],
    )

    loss = in_batch_negative_loss(q_emb, p_emb, h_emb)
    loss.backward()
    optimizer.step()

    print(f'step {step+1}  loss: {loss.item():.4f}')